# Option 2 — Similarity-Weighted Merging

**Hypothesis**: Naïve merging hurts because peer university papers have a different citation distribution to AUB papers.  
Instead of treating every peer paper equally, we **reweight each peer paper** by its cosine similarity to the AUB training corpus in TF-IDF feature space.  
Papers that look like AUB papers get high weight; dissimilar papers get low weight.  
AUB papers always get weight = 1.0.

**Weighting strategies compared**

| Strategy | Description |
|----------|-------------|
| `mean_sim` | Mean cosine similarity to all AUB train papers |
| `top_k_mean` | Mean cosine similarity to the K most-similar AUB papers (K=50) |
| `max_sim` | Max cosine similarity to any AUB train paper |
| `sigmoid` | Sigmoid-scaled mean sim → sharpens the weight distribution |
| `threshold_filter` | Hard cutoff: drop peer papers with mean sim < τ |

**Reference results**

| Model | F1 |
|-------|---------|
| AUB-only baseline | 62.55% |
| Selective domain (best ever) | **63.33%** |
| Naïve merge | 53.56% |

**Input**: `data/processed/all_unis_cleaned.pkl` (output of nb 06)  
**Test set**: AUB-only 2018-2020 (same as baseline for fair comparison)

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path
from scipy.sparse import issparse

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
%matplotlib inline

# ── Reference results ─────────────────────────────────────────────────────────
BASELINE = dict(name='AUB-only baseline',   f1=0.6255, roc_auc=0.8104,
                recall=0.7715, precision=0.5258, threshold=0.54)
BEST     = dict(name='Selective domain',    f1=0.6333)
NAIVE    = dict(name='Naïve merge',         f1=0.5356, roc_auc=0.8240,
                recall=0.7032, precision=0.4323, threshold=0.45)

print('Imports OK')

## 1. Load Merged Data

In [ ]:
data_path = Path('../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found at {data_path}\n"
        "Run notebooks 04 → 05 → 06 first."
    )

df = pd.read_pickle(data_path)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nRows per institution:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")

## 2. Temporal Split

In [ ]:
TRAIN_YEARS = [2015, 2016, 2017]
TEST_YEARS  = [2018, 2019, 2020]

df_train = df[df['Year'].isin(TRAIN_YEARS)].copy()
df_test  = df[(df['Year'].isin(TEST_YEARS)) & (df['institution'] == 'AUB')].copy()

# Also keep an AUB-only train set for the baseline reference
df_train_aub = df_train[df_train['institution'] == 'AUB'].copy()

print("TRAIN (2015-2017):")
print(df_train['institution'].value_counts().to_string())
print(f"\nAUB-only train: {len(df_train_aub):,}")
print(f"TEST (AUB 2018-2020): {len(df_test):,}")

## 3. Feature Engineering

Same pipeline as nb 43: TF-IDF text + venue + author + metadata.  
The TF-IDF vectorizer is **fit on merged train** (so it sees all vocabulary) but  
similarity weights are computed in this shared space.

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    return str(text).lower()

In [ ]:
# Fit on merged train; transform test
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words='english'
)

tfidf_train_sparse = tfidf.fit_transform(
    df_train['Abstract'].apply(preprocess_text)
)
tfidf_test_sparse  = tfidf.transform(
    df_test['Abstract'].apply(preprocess_text)
)

# Also a sparse matrix for AUB-only train (needed for similarity)
aub_mask_train = (df_train['institution'] == 'AUB').values
tfidf_aub_sparse = tfidf_train_sparse[aub_mask_train]

feat_names = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]
print(f"TF-IDF vocab: {len(feat_names)} features")
print(f"AUB train rows for similarity reference: {tfidf_aub_sparse.shape[0]}")

In [ ]:
def build_venue_features(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    col_map = {
        'snip':                 'SNIP (publication year)',
        'snip_percentile':      'SNIP percentile (publication year) *',
        'citescore':            'CiteScore (publication year)',
        'citescore_percentile': 'CiteScore percentile (publication year) *',
        'sjr':                  'SJR (publication year)',
        'sjr_percentile':       'SJR percentile (publication year) *',
    }
    for feat, col in col_map.items():
        vf[feat] = pd.to_numeric(
            subset_df[col] if col in subset_df.columns else np.nan,
            errors='coerce'
        )
    pct_cols = ['snip_percentile', 'citescore_percentile', 'sjr_percentile']
    vf['avg_venue_percentile']  = vf[pct_cols].mean(axis=1)
    vf['is_top_journal']        = (
        (vf['snip_percentile'] >= 90) |
        (vf['citescore_percentile'] >= 90) |
        (vf['sjr_percentile'] >= 90)
    ).astype(int)
    vf['venue_score_composite'] = (
        vf['snip'].fillna(0) * 0.33 +
        vf['citescore'].fillna(0) * 0.33 +
        vf['sjr'].fillna(0) * 0.34
    )
    for col in vf.columns:
        med = vf[col].median()
        vf[col] = vf[col].fillna(0 if pd.isna(med) else med)
    return vf


def build_author_features(subset_df):
    af = pd.DataFrame(index=subset_df.index)
    for feat, col in [('num_authors', 'Number of Authors'),
                       ('num_institutions', 'Number of Institutions'),
                       ('num_countries', 'Number of Countries/Regions')]:
        series = subset_df[col] if col in subset_df.columns else pd.Series(np.nan, index=subset_df.index)
        af[feat] = pd.to_numeric(series, errors='coerce')
    af['is_single_author']        = (af['num_authors'] == 1).astype(int)
    af['is_international_collab'] = (af['num_countries'] > 1).astype(int)
    af['is_multi_institution']    = (af['num_institutions'] > 1).astype(int)
    af['authors_per_institution'] = af['num_authors'] / af['num_institutions'].replace(0, 1)
    af['team_size_small']  = (af['num_authors'] <= 3).astype(int)
    af['team_size_medium'] = ((af['num_authors'] > 3) & (af['num_authors'] <= 10)).astype(int)
    af['team_size_large']  = (af['num_authors'] > 10).astype(int)
    for col in af.columns:
        med = af[col].median()
        af[col] = af[col].fillna(0 if pd.isna(med) else med)
    return af


def build_metadata_features(subset_df, pub_type_cols=None, source_type_cols=None):
    mf = pd.DataFrame(index=subset_df.index)
    mf['is_open_access'] = subset_df.get(
        'Open Access', pd.Series(np.nan, index=subset_df.index)
    ).notna().astype(int)
    mf['topic_prominence'] = pd.to_numeric(
        subset_df.get('Topic Prominence Percentile',
                      pd.Series(np.nan, index=subset_df.index)),
        errors='coerce'
    )
    med_tp = mf['topic_prominence'].median()
    mf['topic_prominence'] = mf['topic_prominence'].fillna(0 if pd.isna(med_tp) else med_tp)
    pub_dummies = pd.get_dummies(
        subset_df.get('Publication type', pd.Series(dtype=str)),
        prefix='pubtype', dummy_na=False
    )
    if pub_type_cols is not None:
        pub_dummies = pub_dummies.reindex(columns=pub_type_cols, fill_value=0)
    src_dummies = pd.get_dummies(
        subset_df.get('Source type', pd.Series(dtype=str)),
        prefix='sourcetype', dummy_na=False
    )
    if source_type_cols is not None:
        src_dummies = src_dummies.reindex(columns=source_type_cols, fill_value=0)
    mf = pd.concat([mf, pub_dummies, src_dummies], axis=1)
    return mf, list(pub_dummies.columns), list(src_dummies.columns)


venue_train  = build_venue_features(df_train)
author_train = build_author_features(df_train)
meta_train, pub_type_cols, source_type_cols = build_metadata_features(df_train)

venue_test  = build_venue_features(df_test)
author_test = build_author_features(df_test)
meta_test, _, _ = build_metadata_features(df_test, pub_type_cols, source_type_cols)

print(f"Venue features:   {venue_train.shape[1]}")
print(f"Author features:  {author_train.shape[1]}")
print(f"Metadata features:{meta_train.shape[1]}")

In [ ]:
# Dense TF-IDF matrices
tfidf_train_df = pd.DataFrame(
    tfidf_train_sparse.toarray(), columns=feat_names, index=df_train.index
)
tfidf_test_df = pd.DataFrame(
    tfidf_test_sparse.toarray(), columns=feat_names, index=df_test.index
)

X_train = pd.concat([tfidf_train_df, venue_train, author_train, meta_train], axis=1)
X_test  = pd.concat([tfidf_test_df,  venue_test,  author_test,  meta_test],  axis=1)

# Targets
citation_threshold = df_train['Citations'].quantile(0.75)
y_train = (df_train['Citations'] >= citation_threshold).astype(int)
y_test  = (df_test['Citations']  >= citation_threshold).astype(int)

print(f"Feature matrix train: {X_train.shape}")
print(f"Feature matrix test:  {X_test.shape}")
print(f"Citation threshold (top 25% of train): {citation_threshold:.0f}")
print(f"Train high-impact: {y_train.mean()*100:.1f}%")
print(f"Test  high-impact: {y_test.mean()*100:.1f}%")

## 4. Compute Similarity Weights

For each paper in the training set we compute its cosine similarity to the AUB sub-corpus.  
AUB papers are anchors (weight = 1.0); peer papers earn weights in **[0, 1]**.

> **Why TF-IDF space?**  
> TF-IDF captures topic similarity — a peer paper about the same medical topic as most AUB papers should be a useful training example.  
> This is the same feature space the model ultimately uses, so high-similarity papers genuinely look like AUB papers to the model.

In [ ]:
print("Computing cosine similarity between peer papers and AUB train corpus...")
print(f"  Peer+AUB train matrix: {tfidf_train_sparse.shape}")
print(f"  AUB reference matrix:  {tfidf_aub_sparse.shape}")

# cosine_similarity returns (n_train, n_aub_train)
# Each row = one training paper; each column = one AUB train paper
sim_matrix = cosine_similarity(tfidf_train_sparse, tfidf_aub_sparse)
print(f"Similarity matrix shape: {sim_matrix.shape}")

# ── Per-paper similarity stats ─────────────────────────────────────────────
sim_mean  = sim_matrix.mean(axis=1)          # mean over all AUB papers
sim_max   = sim_matrix.max(axis=1)           # max  over all AUB papers

K = 50
sim_top_k = np.sort(sim_matrix, axis=1)[:, -K:].mean(axis=1)   # mean of top-K

def sigmoid_scale(x, centre=None, steepness=10):
    """Sigmoid scaled so the median maps to 0.5."""
    if centre is None:
        centre = np.median(x)
    return 1.0 / (1.0 + np.exp(-steepness * (x - centre)))

sim_sigmoid = sigmoid_scale(sim_mean)

print(f"\nSimilarity statistics (all {len(sim_mean)} train papers):")
for name, arr in [('mean_sim', sim_mean), ('top_k_mean', sim_top_k),
                  ('max_sim', sim_max), ('sigmoid', sim_sigmoid)]:
    print(f"  {name:<12}: min={arr.min():.3f}  median={np.median(arr):.3f}  "
          f"mean={arr.mean():.3f}  max={arr.max():.3f}")

In [ ]:
def make_weights(sim_array, institution_col, strategy='mean_sim', threshold=None):
    """
    Build sample_weight array.
    AUB papers → 1.0
    Peer papers → similarity score (or 0 if below threshold)
    """
    weights = sim_array.copy().astype(float)
    # AUB rows keep weight = 1.0 regardless of similarity to themselves
    is_aub = (institution_col == 'AUB').values
    weights[is_aub] = 1.0
    if threshold is not None:
        weights[~is_aub & (weights < threshold)] = 0.0
    return weights

# ── Strategy dict: name → weight array ────────────────────────────────────
strategies = {
    'mean_sim':         make_weights(sim_mean,    df_train['institution']),
    'top_k_mean':       make_weights(sim_top_k,   df_train['institution']),
    'max_sim':          make_weights(sim_max,      df_train['institution']),
    'sigmoid':          make_weights(sim_sigmoid,  df_train['institution']),
    'threshold_0.05':   make_weights(sim_mean,     df_train['institution'], threshold=0.05),
    'threshold_0.10':   make_weights(sim_mean,     df_train['institution'], threshold=0.10),
    'threshold_0.20':   make_weights(sim_mean,     df_train['institution'], threshold=0.20),
}

print("Weight distributions per strategy (peer papers only):")
peer_mask = (df_train['institution'] != 'AUB').values
for name, w in strategies.items():
    peer_w = w[peer_mask]
    n_zero = (peer_w == 0).sum()
    print(f"  {name:<20}: median={np.median(peer_w):.3f}  mean={peer_w.mean():.3f}  "
          f"max={peer_w.max():.3f}  zero={n_zero} ({n_zero/len(peer_w)*100:.1f}%)")

## 5. Visualise Similarity Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'AUB': 'steelblue', 'Lehigh': 'darkorange',
          'Marquette': 'forestgreen', 'Villanova': 'crimson'}

for ax, (strat_name, sim_arr) in zip(axes.flat,
    [('mean_sim', sim_mean), ('top_k_mean', sim_top_k),
     ('max_sim', sim_max),   ('sigmoid', sim_sigmoid)]):

    for inst in df_train['institution'].unique():
        mask = (df_train['institution'] == inst).values
        data = sim_arr[mask]
        if len(np.unique(data)) < 2:
            # Constant array (e.g. AUB weight=1.0) — draw a vertical line instead
            ax.axvline(data[0], color=colors.get(inst, 'grey'), linewidth=2,
                       alpha=0.7, label=f'{inst} (constant={data[0]:.2f})')
        else:
            ax.hist(data, bins=40, alpha=0.55,
                    label=inst, color=colors.get(inst, 'grey'), density=True)

    ax.set_title(f'Strategy: {strat_name}', fontweight='bold')
    ax.set_xlabel('Weight / Similarity')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Similarity weight distributions by institution', fontsize=14, fontweight='bold')
plt.tight_layout()

figures_dir = Path('../reports/figures')
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / 'sim_weight_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Helper: Train + Evaluate

In [ ]:
def find_optimal_threshold(y_true, y_proba, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.90, 0.01)
    f1s = [f1_score(y_true, (y_proba >= t).astype(int)) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    return thresholds[best_idx], f1s[best_idx]


def train_and_evaluate(X_tr, y_tr, X_te, y_te, sample_weight=None, label='model'):
    """Fit logistic regression and return metrics dict."""
    lr = LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced', n_jobs=-1
    )
    lr.fit(X_tr, y_tr, sample_weight=sample_weight)
    proba = lr.predict_proba(X_te)[:, 1]
    opt_t, opt_f1 = find_optimal_threshold(y_te, proba)
    y_pred = (proba >= opt_t).astype(int)
    return {
        'label':     label,
        'f1':        f1_score(y_te, y_pred),
        'roc_auc':   roc_auc_score(y_te, proba),
        'recall':    recall_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred),
        'threshold': opt_t,
        'n_train':   len(y_tr),
        'proba':     proba,
        'model':     lr,
    }

print('Helpers defined.')

## 7. Run All Weighting Strategies

In [ ]:
results = {}

# ── AUB-only baseline (no peer data) ─────────────────────────────────────
aub_only_mask = (df_train['institution'] == 'AUB').values
results['aub_only'] = train_and_evaluate(
    X_train[aub_only_mask], y_train[aub_only_mask],
    X_test, y_test,
    sample_weight=None, label='AUB-only'
)
print(f"AUB-only: F1={results['aub_only']['f1']*100:.2f}%")

# ── Naïve merge (no weighting) ────────────────────────────────────────────
results['naive_merge'] = train_and_evaluate(
    X_train, y_train, X_test, y_test,
    sample_weight=None, label='Naïve merge'
)
print(f"Naïve merge: F1={results['naive_merge']['f1']*100:.2f}%")

# ── Similarity-weighted strategies ────────────────────────────────────────
for strat_name, weights in strategies.items():
    r = train_and_evaluate(
        X_train, y_train, X_test, y_test,
        sample_weight=weights, label=strat_name
    )
    results[strat_name] = r
    delta = (r['f1'] - results['aub_only']['f1']) * 100
    print(f"{strat_name:<22}: F1={r['f1']*100:.2f}%  Δ={delta:+.2f}pp  "
          f"threshold={r['threshold']:.2f}")

print('\nDone.')

## 8. Results Table

In [ ]:
rows = []
for key, r in results.items():
    rows.append({
        'Strategy': r['label'],
        'F1 (%)':  round(r['f1'] * 100, 2),
        'ROC-AUC (%)': round(r['roc_auc'] * 100, 2),
        'Recall (%)':  round(r['recall'] * 100, 2),
        'Precision (%)': round(r['precision'] * 100, 2),
        'Threshold': round(r['threshold'], 2),
        'n_train':  r['n_train'],
    })

# Add historical reference rows
rows.insert(0, {'Strategy': '*** Selective domain (best ever) ***',
                'F1 (%)': 63.33, 'ROC-AUC (%)': np.nan,
                'Recall (%)': np.nan, 'Precision (%)': np.nan,
                'Threshold': 0.54, 'n_train': 2545})

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values('F1 (%)', ascending=False).reset_index(drop=True)

display(results_df)

## 9. Visual Comparison

In [ ]:
# Exclude the reference row from plotting (NaN metrics)
plot_df = results_df.dropna(subset=['ROC-AUC (%)']).copy()

fig, ax = plt.subplots(figsize=(13, 6))

colors_bar = []
for strat in plot_df['Strategy']:
    if 'AUB-only' in strat:
        colors_bar.append('steelblue')
    elif 'Naïve' in strat:
        colors_bar.append('tomato')
    else:
        colors_bar.append('darkorange')

bars = ax.barh(plot_df['Strategy'], plot_df['F1 (%)'],
               color=colors_bar, alpha=0.85, edgecolor='white')

# Reference lines
ax.axvline(BEST['f1'] * 100, color='green', linestyle='--', linewidth=1.5,
           label=f'Best ever (63.33% — selective domain)')
ax.axvline(BASELINE['f1'] * 100, color='steelblue', linestyle=':', linewidth=1.5,
           label=f'AUB-only baseline (62.55%)')
ax.axvline(NAIVE['f1'] * 100, color='tomato', linestyle=':', linewidth=1.5,
           label=f'Naïve merge (53.56%)')

# Value labels
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{w:.2f}%', va='center', fontsize=9)

ax.set_xlabel('F1 Score (%)', fontsize=12)
ax.set_title('Similarity-Weighted Merging — All Strategies vs References',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(45, 70)

plt.tight_layout()
plt.savefig(figures_dir / 'sim_weighted_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# F1 / ROC-AUC / Recall / Precision side-by-side for key strategies
key_strategies = ['AUB-only', 'Naïve merge']
# Add best-performing sim-weighted strategy
sim_rows = plot_df[~plot_df['Strategy'].isin(key_strategies)].sort_values('F1 (%)', ascending=False)
if len(sim_rows) > 0:
    key_strategies.append(sim_rows.iloc[0]['Strategy'])
    if len(sim_rows) > 1:
        key_strategies.append(sim_rows.iloc[1]['Strategy'])

key_df = plot_df[plot_df['Strategy'].isin(key_strategies)].copy()

metrics_to_plot = ['F1 (%)', 'ROC-AUC (%)', 'Recall (%)', 'Precision (%)']
x = np.arange(len(metrics_to_plot))
width = 0.8 / len(key_df)
offsets = np.linspace(-(0.8 - width) / 2, (0.8 - width) / 2, len(key_df))

palette = plt.cm.Set2.colors

fig, ax = plt.subplots(figsize=(12, 6))
for i, (_, row) in enumerate(key_df.iterrows()):
    vals = [row[m] / 100 for m in metrics_to_plot]
    bars = ax.bar(x + offsets[i], vals, width,
                  label=row['Strategy'], color=palette[i % len(palette)], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h*100:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Key Strategies — Metric Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'sim_weighted_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Weight vs Performance Deep-Dive

Does giving peer papers *any* positive weight add noise, or does it add signal?  
We sweep `alpha` — a blending scalar that controls how strongly we apply the weights:

```
effective_weight = alpha * sim_weight + (1 - alpha) * 1.0   (peer papers)
```

At alpha=0, all papers (AUB + peer) have weight 1.0 → naïve merge.  
At alpha=1, peer papers get pure similarity weight → Option 2 as designed.

In [ ]:
alphas = np.arange(0.0, 1.05, 0.1)
best_strategy_name = sim_rows.iloc[0]['Strategy'] if len(sim_rows) > 0 else 'mean_sim'
best_strategy_weights = strategies.get(best_strategy_name, strategies['mean_sim'])

peer_mask_train = (df_train['institution'] != 'AUB').values
alpha_f1s = []

for alpha in alphas:
    w = np.ones(len(df_train))
    w[peer_mask_train] = (
        alpha * best_strategy_weights[peer_mask_train]
        + (1 - alpha) * 1.0
    )
    r = train_and_evaluate(
        X_train, y_train, X_test, y_test,
        sample_weight=w, label=f'alpha={alpha:.1f}'
    )
    alpha_f1s.append(r['f1'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alphas, [f * 100 for f in alpha_f1s],
        marker='o', color='darkorange', linewidth=2, markersize=7,
        label=f'Sim-weighted ({best_strategy_name})')
ax.axhline(BASELINE['f1'] * 100, color='steelblue', linestyle='--',
           label=f'AUB-only baseline ({BASELINE["f1"]*100:.2f}%)')
ax.axhline(NAIVE['f1'] * 100, color='tomato', linestyle='--',
           label=f'Naïve merge ({NAIVE["f1"]*100:.2f}%)')
ax.axhline(BEST['f1'] * 100, color='green', linestyle=':',
           label=f'Best ever ({BEST["f1"]*100:.2f}%)')

best_alpha_idx = int(np.argmax(alpha_f1s))
ax.annotate(
    f'Peak: {alpha_f1s[best_alpha_idx]*100:.2f}% at α={alphas[best_alpha_idx]:.1f}',
    xy=(alphas[best_alpha_idx], alpha_f1s[best_alpha_idx] * 100),
    xytext=(alphas[best_alpha_idx] + 0.05, alpha_f1s[best_alpha_idx] * 100 + 0.5),
    fontsize=10, color='darkorange',
    arrowprops=dict(arrowstyle='->', color='darkorange')
)

ax.set_xlabel('α (weight blending factor)', fontsize=12)
ax.set_ylabel('F1 Score (%)', fontsize=12)
ax.set_title(f'F1 vs α — Similarity weight blending ({best_strategy_name})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / 'sim_weighted_alpha_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best alpha: {alphas[best_alpha_idx]:.1f}  →  F1={alpha_f1s[best_alpha_idx]*100:.2f}%")

## 11. Per-Strategy Threshold Curves

In [ ]:
thresholds = np.arange(0.10, 0.90, 0.01)

# Collect F1-vs-threshold for each strategy
f1_curves = {}
for key, r in results.items():
    proba = r['proba']
    f1_curves[r['label']] = [
        f1_score(y_test, (proba >= t).astype(int)) for t in thresholds
    ]

fig, ax = plt.subplots(figsize=(12, 6))

palette_lines = plt.cm.tab10.colors
for i, (label, curve) in enumerate(f1_curves.items()):
    lw = 2.5 if label in ('AUB-only', 'Naïve merge') else 1.2
    ls = '-'  if label in ('AUB-only', 'Naïve merge') else '--'
    ax.plot(thresholds, curve, linewidth=lw, linestyle=ls,
            color=palette_lines[i % len(palette_lines)], label=label)

ax.axhline(BEST['f1'], color='green', linestyle=':', linewidth=1.5,
           label=f'Best ever (63.33%)')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 vs Threshold — All Strategies', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower center', ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / 'sim_weighted_threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Confusion Matrix — Best Strategy

In [ ]:
# Identify the best sim-weighted strategy by F1
sim_results = {k: v for k, v in results.items()
               if k not in ('aub_only', 'naive_merge')}
best_key    = max(sim_results, key=lambda k: sim_results[k]['f1'])
best_result = sim_results[best_key]

print(f"Best similarity-weighted strategy: {best_key}")
print(f"  F1={best_result['f1']*100:.2f}%  ROC-AUC={best_result['roc_auc']*100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, key, title_suffix in [
    (axes[0], 'aub_only',    'AUB-only baseline'),
    (axes[1], 'naive_merge', 'Naïve merge'),
    (axes[2], best_key,      f'Best sim-weighted\n({best_key})'),
]:
    r = results[key]
    y_pred = (r['proba'] >= r['threshold']).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
    ax.set_title(
        f"{title_suffix}\nF1={r['f1']*100:.2f}%  t={r['threshold']:.2f}",
        fontweight='bold', fontsize=10
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(figures_dir / 'sim_weighted_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Summary & Conclusions

In [ ]:
aub_f1   = results['aub_only']['f1']
naive_f1 = results['naive_merge']['f1']
best_sim_f1 = best_result['f1']
delta_vs_aub   = (best_sim_f1 - aub_f1) * 100
delta_vs_naive = (best_sim_f1 - naive_f1) * 100
delta_vs_best  = (best_sim_f1 - BEST['f1']) * 100

print("=" * 65)
print("SIMILARITY-WEIGHTED MERGING — RESULTS SUMMARY")
print("=" * 65)
print()
print(f"{'Model':<38} {'F1':>8}")
print("-" * 48)
print(f"{'Selective domain (best ever)':38} {BEST['f1']*100:>7.2f}%")
print(f"{'AUB-only baseline':38} {aub_f1*100:>7.2f}%")
print(f"{'Naïve merge':38} {naive_f1*100:>7.2f}%")
print("-" * 48)
print(f"{'Best sim-weighted (' + best_key + ')':38} {best_sim_f1*100:>7.2f}%")
print()
print(f"Δ vs AUB-only baseline:       {delta_vs_aub:+.2f} pp")
print(f"Δ vs naïve merge:             {delta_vs_naive:+.2f} pp")
print(f"Δ vs best ever (63.33%):      {delta_vs_best:+.2f} pp")
print()
if delta_vs_aub > 1.0:
    print("CONCLUSION: Similarity weighting successfully bridges the gap.")
    print("Peer data, when properly weighted, improves AUB predictions.")
elif delta_vs_aub > 0:
    print("CONCLUSION: Marginal improvement over AUB-only baseline.")
    print("Similarity weighting partially recovers the loss from naïve merging,")
    print("but the gain is small — institution-specific signals dominate.")
elif delta_vs_naive > 5:
    print("CONCLUSION: Similarity weighting substantially outperforms naïve merging.")
    print("However, it still falls short of the AUB-only baseline.")
    print("Peer data cannot fully replace AUB-specific training signals.")
else:
    print("CONCLUSION: Similarity weighting does not recover naïve-merge losses.")
    print("Even re-weighted peer data introduces more noise than signal.")
    print("Recommendation: stop Option 2 and rely on AUB-only (Option 1).")

## 14. Save Artifacts

In [ ]:
models_dir  = Path('../models')
metrics_dir = Path('../reports/metrics')
models_dir.mkdir(exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

# Save best weighted model
with open(models_dir / 'sim_weighted_best_model.pkl', 'wb') as f:
    pickle.dump(best_result['model'], f)

with open(models_dir / 'sim_weighted_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save results CSV
results_df.to_csv(metrics_dir / 'sim_weighted_results.csv', index=False)

print("Saved:")
print("  models/sim_weighted_best_model.pkl")
print("  models/sim_weighted_tfidf.pkl")
print("  reports/metrics/sim_weighted_results.csv")
print("  reports/figures/sim_weight_distributions.png")
print("  reports/figures/sim_weighted_comparison.png")
print("  reports/figures/sim_weighted_metrics_comparison.png")
print("  reports/figures/sim_weighted_alpha_sweep.png")
print("  reports/figures/sim_weighted_threshold_curves.png")
print("  reports/figures/sim_weighted_confusion_matrices.png")